<a href="https://colab.research.google.com/github/elkins-lab/synth-saxs/blob/main/examples/interactive_tutorials/hydration_shell_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive Hydration Shell Analysis

In Small-Angle X-ray Scattering (SAXS), the protein is not a "dry" object in a vacuum. It is surrounded by solvent (water). One of the most subtle but important effects in SAXS modeling is the **hydration shell**—a layer of water molecules immediately surrounding the protein that is slightly denser (~2-5%) than bulk water.

This tutorial demonstrates how this "excess" density affects the scattering profile and the perceived size of the protein.

In [ ]:
import sys

# Install dependencies if running in Colab or a new environment
if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/elkins-lab/synth-saxs.git biotite matplotlib ipywidgets
else:
    sys.path.append("../../")

In [ ]:
import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

from synth_saxs import calculate_radius_of_gyration, calculate_saxs_profile

print("Setup complete.")

## 1. Load a Reference Structure
We will use **Ubiquitin (1UBQ)**, a small and well-studied protein.

In [ ]:
# Download and load 1UBQ
pdb_file = rcsb.fetch("1UBQ", "pdb", ".")
structure = strucio.load_structure(pdb_file)
if hasattr(structure, "stack_depth") and structure.stack_depth() > 1:
    structure = structure[0]

# Pre-calculate the Radius of Gyration (dry)
rg_dry = calculate_radius_of_gyration(structure)
print(f"Structure loaded. Dry Rg: {rg_dry:.2f} A")

## 2. Interactive Visualization

The hydration shell is modeled by adding an excess density $\Delta\rho_{shell}$ to the solvent term. 

The effective scattering factor $f_{eff}(q)$ for an atom is roughly:
$$ f_{eff}(q) = f_{vac}(q) - (\rho_{sol} - \Delta\rho_{shell}) \cdot V \cdot \exp(-q^2 R^2 / 10) $$

As $\Delta\rho_{shell}$ increases, the "contrast" between the protein and its immediate surroundings changes, making the protein appear larger or "brighter" to X-rays.

In [ ]:
# The bulk profile does not change with the slider, so compute it once.
q_bulk, i_bulk = calculate_saxs_profile(structure, hydration_shell_density=0.0, n_points=100)


def update_plot(shell_density):
    # 1. Calculate profiles
    q = q_bulk
    _, i_shell = calculate_saxs_profile(
        structure, hydration_shell_density=shell_density, n_points=100
    )

    # 2. Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Log-linear plot
    ax1.semilogy(q, i_bulk, "k--", label="Bulk Solvent only", alpha=0.5)
    ax1.semilogy(q, i_shell, "r-", linewidth=2, label=f"Shell Density: {shell_density:.3f} e/A^3")
    ax1.set_xlabel("q (A^-1)", fontsize=12)
    ax1.set_ylabel("log I(q)", fontsize=12)
    ax1.set_title("Effect on Scattering Intensity", fontsize=14)
    ax1.legend()

    # Difference plot (normalized to bulk at q=0)
    diff = (i_shell - i_bulk) / i_bulk[0] * 100
    ax2.plot(q, diff, "g-")
    ax2.set_xlabel("q (A^-1)", fontsize=12)
    ax2.set_ylabel("% Difference (relative to I(0))", fontsize=12)
    ax2.set_title("Relative Intensity Increase", fontsize=14)
    ax2.grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()


# Create the interactive slider
interact(
    update_plot,
    shell_density=FloatSlider(
        value=0.03,
        min=0.0,
        max=0.06,
        step=0.005,
        description="Shell Dens:",
        continuous_update=False,
    ),
);

### What are you seeing?

1. **Increased Contrast:** Adding a hydration shell increases the scattering intensity at $q=0$. This is because the "effective volume" of the protein is increasing as it pulls in a layer of denser water.
2. **The "Boom" at low q:** Notice that the relative difference is most pronounced at low $q$ values. This is where the global shape (the "envelope") of the protein dominates the scattering.
3. **Experimental Reality:** In real experiments, the hydration shell density is typically between **0.02 and 0.05 e/Å³**. If you ignore this effect, your simulated curves will often systematically underestimate the experimental intensity at low angles.

## 3. The Mathematical Cost of Ignoring Hydration
We can mathematically demonstrate the importance of the hydration shell by using the `synth_saxs.fitting` module.
Let's pretend our `shell_density=0.03` profile is the **real experimental data**. What happens if we try to fit a **dry** theoretical profile (density=0.0) to it?

In [ ]:
import numpy as np

from synth_saxs.fitting import fit_profile

# 1. Create the 'Experimental' Target (wet, with 2% noise)
q_exp, i_wet = calculate_saxs_profile(structure, hydration_shell_density=0.03, n_points=100)
noise = 0.02 * i_wet
i_exp = np.random.normal(i_wet, noise)
err_exp = noise

# 2. Calculate the 'Theoretical' Profile (dry)
q_calc, i_dry = calculate_saxs_profile(structure, hydration_shell_density=0.0, n_points=100)

# 3. Fit dry to wet
c, k, chi_sq = fit_profile(i_exp, err_exp, i_dry)
i_fit = c * i_dry + k

print(f"Fit Results: c = {c:.2f}, k = {k:.2f}, chi^2 = {chi_sq:.2f}")

# 4. Visualize the mismatch
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), height_ratios=[3, 1], sharex=True)
ax1.errorbar(
    q_exp, i_exp, yerr=err_exp, fmt="ko", markersize=4, alpha=0.5, label="Experimental (Wet)"
)
ax1.semilogy(q_calc, i_fit, "r-", linewidth=2, label="Fitted Theoretical (Dry)")
ax1.set_ylabel("log I(q)", fontsize=12)
ax1.set_title(f"Fitting Dry Model to Wet Data (chi^2 = {chi_sq:.1f})", fontsize=14)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

residuals = (i_exp - i_fit) / err_exp
ax2.plot(q_exp, residuals, "k.", markersize=4, alpha=0.5)
ax2.axhline(0, color="r", linestyle="--", lw=1)
ax2.set_xlabel("q (A^-1)", fontsize=12)
ax2.set_ylabel("ΔI / σ", fontsize=12)
ax2.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(
    "A chi^2 > 1.5 indicates a poor fit. The massive U-shape in the residuals at low q proves that ignoring hydration systematically misrepresents the size of the protein!"
)